# Random Forest sensitivity check

In [ ]:
#mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception as exc:
    IN_COLAB = False
    print('Not running in Colab, or Drive mount skipped:', exc)

Mounted at /content/drive


In [ ]:
#Configures
from __future__ import annotations
from itertools import product
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
RANDOM_SEED = 42
TEST_SIZE = 0.3
CLASS_LABELS = [
    'Winter wheat',
    'Winter barley',
    'Spring barley',
    'Beet (sugar beet / fodder beet)',
    'Maize',
    'Oilseed rape',
    'Potatoes',
    'Pulses / field beans and peas',
]
SEASONAL_WINDOWS = ['winter_establishment', 'spring_growth', 'summer_peak', 'late_season']
S2_BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
S2_INDICES = ['NDVI', 'NDRE', 'LSWI', 'EVI']
S1_BASE_FEATURES = ['VV', 'VH', 'VV_minus_VH', 'VV_div_VH']
S2_FEATURE_BANDS = [
    f'{season}_{band}'
    for season in SEASONAL_WINDOWS
    for band in (S2_BANDS + S2_INDICES)
]
S1_FEATURE_BANDS = [
    f'{season}_{band}'
    for season in SEASONAL_WINDOWS
    for band in S1_BASE_FEATURES
]
PARAMETER_GRID = {
    'n_estimators': [300, 500, 800],
    'max_features': ['sqrt', 'log2'],
    'min_samples_leaf': [1, 3, 5],
    'class_weight': ['balanced', 'balanced_subsample'],
}
PROJECT_DIR = Path('/content/drive/MyDrive/Dissertation') if IN_COLAB else Path.cwd().parents[1]
RUN_DIR = PROJECT_DIR / '5_rf_sensitivity_check'
INPUT_CSV = PROJECT_DIR / '2_sentinel1_sentinel2_model' / 'data' / 'processed' / 's1s2_2022_east_anglia_crome_8class_features.csv'
TABLE_DIR = RUN_DIR / 'outputs' / 'tables'
FIGURE_DIR = RUN_DIR / 'outputs' / 'figures'
LOG_DIR = RUN_DIR / 'outputs' / 'logs'
RUN_MANIFEST = LOG_DIR / 'rf_sensitivity_run_manifest.json'
for folder in [TABLE_DIR, FIGURE_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print('PROJECT_DIR:', PROJECT_DIR)
print('RUN_DIR:', RUN_DIR)
print('INPUT_CSV:', INPUT_CSV)


PROJECT_DIR: /content/drive/MyDrive/Dissertation
RUN_DIR: /content/drive/MyDrive/Dissertation/5_rf_sensitivity_check
INPUT_CSV: /content/drive/MyDrive/Dissertation/2_sentinel1_sentinel2_model/data/processed/s1s2_2022_east_anglia_crome_8class_features.csv


In [ ]:
#Load and clean the combined feature table
if not INPUT_CSV.exists():
    raise FileNotFoundError(f'Input feature table not found: {INPUT_CSV}')
df = pd.read_csv(INPUT_CSV)
s2_cols = [col for col in S2_FEATURE_BANDS if col in df.columns]
s1_cols = [col for col in S1_FEATURE_BANDS if col in df.columns]
missing_s2 = sorted(set(S2_FEATURE_BANDS) - set(s2_cols))
missing_s1 = sorted(set(S1_FEATURE_BANDS) - set(s1_cols))
if missing_s2 or missing_s1:
    raise ValueError({'missing_s2': missing_s2, 'missing_s1': missing_s1})
model_df = df[df['analysis_class'].isin(CLASS_LABELS)].copy()
for col in s2_cols + s1_cols:
    model_df[col] = pd.to_numeric(model_df[col], errors='coerce')
model_df = model_df.dropna(subset=['analysis_class', *s2_cols, *s1_cols]).reset_index(drop=True)
print('Clean modelling rows:', len(model_df))
print('Sentinel-2 predictors:', len(s2_cols))
print('Sentinel-1 predictors:', len(s1_cols))
display(model_df['analysis_class'].value_counts().reindex(CLASS_LABELS))

Clean modelling rows: 5742
Sentinel-2 predictors: 56
Sentinel-1 predictors: 16


,count
analysis_class,
Winter wheat,699
Winter barley,732
Spring barley,738
Beet (sugar beet / fodder beet),744
Maize,722
Oilseed rape,691
Potatoes,709
Pulses / field beans and peas,707


In [ ]:
#create a fixed stratified train/test split
train_idx, test_idx = train_test_split(
    model_df.index.to_numpy(),
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=model_df['analysis_class'],
)
train_df = model_df.loc[train_idx].copy()
test_df = model_df.loc[test_idx].copy()
def iter_parameter_settings():
    keys = list(PARAMETER_GRID)
    for idx, values in enumerate(product(*(PARAMETER_GRID[key] for key in keys)), start=1):
        setting = dict(zip(keys, values))
        setting['setting_id'] = f'rf_{idx:02d}'
        yield setting
def run_one_model(train_df, test_df, feature_cols, model_name, setting):
    rf = RandomForestClassifier(
        n_estimators=int(setting['n_estimators']),
        max_features=str(setting['max_features']),
        min_samples_leaf=int(setting['min_samples_leaf']),
        class_weight=str(setting['class_weight']),
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    rf.fit(train_df[feature_cols], train_df['analysis_class'])
    pred = rf.predict(test_df[feature_cols])
    overall = {
        'setting_id': setting['setting_id'],
        'model': model_name,
        'n_estimators': setting['n_estimators'],
        'max_features': setting['max_features'],
        'min_samples_leaf': setting['min_samples_leaf'],
        'class_weight': setting['class_weight'],
        'feature_count': len(feature_cols),
        'train_rows': len(train_df),
        'test_rows': len(test_df),
        'overall_agreement': accuracy_score(test_df['analysis_class'], pred),
        'balanced_accuracy': balanced_accuracy_score(test_df['analysis_class'], pred),
        'macro_f1': f1_score(test_df['analysis_class'], pred, labels=CLASS_LABELS, average='macro', zero_division=0),
        'weighted_f1': f1_score(test_df['analysis_class'], pred, labels=CLASS_LABELS, average='weighted', zero_division=0),
        'reference_note': REFERENCE_NOTE,
        'analysis_note': ANALYSIS_NOTE,
    }
    report = classification_report(
        test_df['analysis_class'], pred, labels=CLASS_LABELS, output_dict=True, zero_division=0
    )
    class_metrics = (
        pd.DataFrame(report)
        .T.loc[CLASS_LABELS, ['precision', 'recall', 'f1-score', 'support']]
        .reset_index()
        .rename(columns={'index': 'class', 'precision': 'user_accuracy', 'recall': 'producer_accuracy', 'f1-score': 'f1_score'})
    )
    for key in ['setting_id', 'n_estimators', 'max_features', 'min_samples_leaf', 'class_weight']:
        class_metrics[key] = setting[key]
    class_metrics['model'] = model_name
    class_metrics['reference_note'] = REFERENCE_NOTE
    return overall, class_metrics
print('Train rows:', len(train_df))
print('Test rows:', len(test_df))
print('Settings:', len(list(iter_parameter_settings())))

Train rows: 4019
Test rows: 1723
Settings: 36


In [ ]:
#run each RF setting for both feature sets.
overall_records = []
class_frames = []
for setting in iter_parameter_settings():
    for model_name, feature_cols in [('S2-only', s2_cols), ('S1+S2', s2_cols + s1_cols)]:
        print(f"Running {setting['setting_id']} {model_name}")
        overall, class_metrics = run_one_model(train_df, test_df, feature_cols, model_name, setting)
        overall_records.append(overall)
        class_frames.append(class_metrics)
overall_metrics = pd.DataFrame(overall_records)
class_metrics = pd.concat(class_frames, ignore_index=True)
shared_cols = ['setting_id', 'n_estimators', 'max_features', 'min_samples_leaf', 'class_weight']
metric_cols = ['overall_agreement', 'balanced_accuracy', 'macro_f1', 'weighted_f1']
s2 = overall_metrics[overall_metrics['model'] == 'S2-only'].copy()
s1s2 = overall_metrics[overall_metrics['model'] == 'S1+S2'].copy()
delta = s1s2[shared_cols + metric_cols].merge(
    s2[['setting_id', *metric_cols]],
    on='setting_id',
    suffixes=('_s1s2', '_s2_only'),
    validate='one_to_one',
)
for metric in metric_cols:
    delta[f'{metric}_delta'] = delta[f'{metric}_s1s2'] - delta[f'{metric}_s2_only']
    delta[f's1s2_better_{metric}'] = delta[f'{metric}_delta'] > 0
s2_class = class_metrics[class_metrics['model'] == 'S2-only'].copy()
s1s2_class = class_metrics[class_metrics['model'] == 'S1+S2'].copy()
class_delta = s1s2_class[[*shared_cols, 'class', 'f1_score', 'user_accuracy', 'producer_accuracy', 'support']].merge(
    s2_class[['setting_id', 'class', 'f1_score', 'user_accuracy', 'producer_accuracy', 'support']],
    on=['setting_id', 'class'],
    suffixes=('_s1s2', '_s2_only'),
    validate='one_to_one',
)
for metric in ['f1_score', 'user_accuracy', 'producer_accuracy']:
    class_delta[f'{metric}_delta'] = class_delta[f'{metric}_s1s2'] - class_delta[f'{metric}_s2_only']
class_delta_summary = (
    class_delta.groupby('class', as_index=False)
    .agg(
        mean_f1_delta=('f1_score_delta', 'mean'),
        min_f1_delta=('f1_score_delta', 'min'),
        max_f1_delta=('f1_score_delta', 'max'),
        settings_with_positive_f1_delta=('f1_score_delta', lambda s: int((s > 0).sum())),
        settings_tested=('f1_score_delta', 'size'),
    )
    .sort_values('mean_f1_delta', ascending=False)
)
print('Run complete')
display(delta[['setting_id', 'macro_f1_delta', 'overall_agreement_delta']].describe())

Running rf_01 S2-only
Running rf_01 S1+S2
Running rf_02 S2-only
Running rf_02 S1+S2
Running rf_03 S2-only
Running rf_03 S1+S2
Running rf_04 S2-only
Running rf_04 S1+S2
Running rf_05 S2-only
Running rf_05 S1+S2
Running rf_06 S2-only
Running rf_06 S1+S2
Running rf_07 S2-only
Running rf_07 S1+S2
Running rf_08 S2-only
Running rf_08 S1+S2
Running rf_09 S2-only
Running rf_09 S1+S2
Running rf_10 S2-only
Running rf_10 S1+S2
Running rf_11 S2-only
Running rf_11 S1+S2
Running rf_12 S2-only
Running rf_12 S1+S2
Running rf_13 S2-only
Running rf_13 S1+S2
Running rf_14 S2-only
Running rf_14 S1+S2
Running rf_15 S2-only
Running rf_15 S1+S2
Running rf_16 S2-only
Running rf_16 S1+S2
Running rf_17 S2-only
Running rf_17 S1+S2
Running rf_18 S2-only
Running rf_18 S1+S2
Running rf_19 S2-only
Running rf_19 S1+S2
Running rf_20 S2-only
Running rf_20 S1+S2
Running rf_21 S2-only
Running rf_21 S1+S2
Running rf_22 S2-only
Running rf_22 S1+S2
Running rf_23 S2-only
Running rf_23 S1+S2
Running rf_24 S2-only
Running rf_2

,macro_f1_delta,overall_agreement_delta
count,36.000000,36.000000
mean,0.080834,0.079948
std,0.004750,0.004392
min,0.071344,0.071387
25%,0.078026,0.076611
50%,0.081517,0.080963
75%,0.084282,0.082995
max,0.088893,0.087638


In [ ]:
#Save metrics and comparison figures.
overall_metrics.to_csv(TABLE_DIR / 'rf_sensitivity_overall_metrics.csv', index=False)
delta.to_csv(TABLE_DIR / 'rf_sensitivity_delta_summary.csv', index=False)
class_metrics.to_csv(TABLE_DIR / 'rf_sensitivity_class_metrics.csv', index=False)
class_delta.to_csv(TABLE_DIR / 'rf_sensitivity_class_delta_metrics.csv', index=False)
class_delta_summary.to_csv(TABLE_DIR / 'rf_sensitivity_class_delta_summary.csv', index=False)
plot_df = delta.sort_values('macro_f1_delta').copy()
labels = plot_df.apply(
    lambda row: f"{row['setting_id']} | n={row['n_estimators']}, {row['max_features']}, leaf={row['min_samples_leaf']}, {str(row['class_weight']).replace('_', ' ')}",
    axis=1,
)
for metric, label, filename in [
    ('macro_f1_delta', 'S1+S2 minus S2-only macro-F1', 'rf_sensitivity_macro_f1_delta.png'),
    ('overall_agreement_delta', 'S1+S2 minus S2-only overall agreement', 'rf_sensitivity_overall_agreement_delta.png'),
]:
    fig_height = max(8, 0.27 * len(plot_df))
    plt.figure(figsize=(11, fig_height))
    colors = np.where(plot_df[metric] >= 0, '#277da1', '#b53d3a')
    plt.barh(range(len(plot_df)), plot_df[metric], color=colors)
    plt.yticks(range(len(plot_df)), labels, fontsize=7)
    plt.axvline(0, color='#222222', linewidth=0.8)
    plt.xlabel(label)
    plt.title(label + ' across Random Forest sensitivity settings')
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / filename, dpi=300)
    plt.show()
# Record the fixed split and tested parameter grid.
run_manifest = {
    'input_feature_table': str(INPUT_CSV),
    'rows_used': int(len(model_df)),
    'feature_counts': {
        'sentinel_2': int(len(s2_cols)),
        'sentinel_1': int(len(s1_cols)),
    },
    'split': {
        'test_size': TEST_SIZE,
        'stratified_by': 'analysis_class',
        'random_seed': RANDOM_SEED,
    },
    'parameter_grid': PARAMETER_GRID,
    'settings_tested': int(len(delta)),
    'positive_macro_f1_deltas': int(delta['s1s2_better_macro_f1'].sum()),
    'positive_overall_agreement_deltas': int(delta['s1s2_better_overall_agreement'].sum()),
    'analysis_note': ANALYSIS_NOTE,
    'reference_note': REFERENCE_NOTE,
}
RUN_MANIFEST.write_text(json.dumps(run_manifest, indent=2), encoding='utf-8')

print('Saved tables to:', TABLE_DIR)
print('Saved figures to:', FIGURE_DIR)


In [ ]:
#Check outputs
expected_outputs = [
    TABLE_DIR / 'rf_sensitivity_overall_metrics.csv',
    TABLE_DIR / 'rf_sensitivity_delta_summary.csv',
    TABLE_DIR / 'rf_sensitivity_class_metrics.csv',
    TABLE_DIR / 'rf_sensitivity_class_delta_metrics.csv',
    TABLE_DIR / 'rf_sensitivity_class_delta_summary.csv',
    FIGURE_DIR / 'rf_sensitivity_macro_f1_delta.png',
    FIGURE_DIR / 'rf_sensitivity_overall_agreement_delta.png',
    RUN_MANIFEST,
]
check = pd.DataFrame({
    'path': [str(p) for p in expected_outputs],
    'exists': [p.exists() for p in expected_outputs],
    'size_bytes': [p.stat().st_size if p.exists() else 0 for p in expected_outputs],
})
display(check)
display(delta.sort_values('macro_f1_delta', ascending=False).head(10))
display(class_delta_summary)
